# Day 8 - LLM foundations & API integration practical tasks
***

These tasks cover calling OpenAI API's and prompting.

## Explore how different model parameters and different prompts influence the response.
#### Here are some guidelines and advice you should consider:
* You can add as many cells as you need for each task.
* Use Markdown cells for your conclusions.
* For tasks that do not specify a model, you may choose any of the two available models: `gpt-5.6-luna` / `gpt-5.4-mini`.
* Within each comparison task, change only the model, prompt, or parameter specified in the instructions. Keep everything else - including the model, prompt, and other settings - the same across calls so you can compare the results fairly. For example, when testing different `temperature` values, use the same model, prompt, and output token limit (`max_output_tokens` for the Responses API or `max_completion_tokens` for the Chat Completions API).
* You can use both - [Responses API and Chat Completions API](https://medium.com/@sanathgoutham/openai-responses-api-vs-chat-completions-what-actually-changed-eeee86aff216) - to complete these tasks, but some model parameters are not supported by Responses API.
* LLMs understand [Markdown](https://www.markdownguide.org/cheat-sheet/) formatting in prompts. For example, wrapping text in double asterisks `(**important text**)` marks it as **bold** and can help signal emphasis.
* To copy text from a Markdown cell, open it in edit mode by clicking the pencil icon or double-clicking the cell.
* Optionally, paste API responses into a Markdown cell for better readability.
* Be creative and experiment.

**Setup (VS Code).** Select the Python 3.12 kernel, make sure `.env` (with `OPENAI_API_KEY`) is in this folder, and run the next cell once.

In [4]:
%pip install -q openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ.pop("OPENAI_API_KEY", None)
load_dotenv()

True

In [6]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in the W2D3 .env file before running this cell.")

client = OpenAI(base_url="https://ds-ai-internship.openai.azure.com/openai/v1/",
                api_key=OPENAI_API_KEY)

**Helper functions.** `ask()` makes one Chat Completions call and returns the response and the time it took. `show()` prints the answers and the token usage. The other helpers extract lists and code from the answers. Run this cell before the tasks.

In [7]:
import time, re, json, math, io, contextlib
from collections import Counter
from IPython.display import Markdown, display
from openai import RateLimitError, APIConnectionError, APITimeoutError

LUNA = "gpt-5.6-luna"
MINI = "gpt-5.4-mini"

def ask(prompt=None, *, messages=None, model=MINI, retries=3, **params):
    """One Chat Completions call. Returns (response, seconds). Anything in params goes straight to the API."""
    if messages is None:
        messages = [{"role": "user", "content": prompt}]
    for attempt in range(retries):
        start = time.perf_counter()
        try:
            resp = client.chat.completions.create(model=model, messages=messages, **params)
            return resp, time.perf_counter() - start
        except (RateLimitError, APIConnectionError, APITimeoutError):
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

def texts(resp):
    return [(c.message.content or "").strip() for c in resp.choices]

def show(resp, seconds=None, label=None):
    if label:
        print(f"##### {label}")
    for c in resp.choices:
        print(f"--- answer {c.index + 1} (finish_reason: {c.finish_reason}) ---")
        print((c.message.content or "").strip(), "\n")
    u = resp.usage
    timing = f" | time: {seconds:.2f}s" if seconds is not None else ""
    print(f"tokens: input={u.prompt_tokens}, output={u.completion_tokens}, total={u.total_tokens}{timing}\n")

def list_items(text):
    """Return the items of a numbered or bulleted list, without the markers and bold marks."""
    items = []
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"^(\d+[.)]|[-*\u2022])\s+", line):
            items.append(re.sub(r"^(\d+[.)]|[-*\u2022])\s+", "", line).replace("**", "").strip())
    return items

def extract_code(text):
    """Return the contents of the first fenced code block (or the whole text if there is none)."""
    m = re.search(r"```(?:python|py)?\s*\n(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

def run_prompt_set(prompts, **extra):
    """Run several prompts with identical settings and print each answer."""
    settings = dict(reasoning_effort="none", temperature=0.7, max_completion_tokens=500)
    settings.update(extra)
    for label, prompt in prompts.items():
        print("PROMPT:", prompt)
        resp, secs = ask(prompt, **settings)
        show(resp, secs, label)

#### 1. Write a Python script to call the OpenAI API and generate a response to the prompt below.

Prompt: `"What is artificial intelligence?"`

Use model `gpt-5.6-luna`.

In [8]:
resp, secs = ask("What is artificial intelligence?", model=LUNA)
show(resp, secs, "gpt-5.6-luna")

##### gpt-5.6-luna
--- answer 1 (finish_reason: stop) ---
Artificial intelligence (AI) is the field of creating computer systems that can perform tasks that typically require human intelligence, such as:

- Understanding language
- Recognizing images and speech
- Learning from data
- Solving problems
- Making predictions or decisions
- Generating text, images, music, or code

AI systems can be rule-based, but many modern systems use machine learning, where they identify patterns in large amounts of data and improve their performance through training. Examples include virtual assistants, recommendation systems, self-driving features, fraud detection, and chatbots. 

tokens: input=11, output=117, total=128 | time: 3.37s



**Notes.** The text of the answer sits in `resp.choices[0].message.content`, and `resp.usage` tells me how many tokens went in and came out. I use both of those in the later tasks.

#### 2. Call the OpenAI API with different models

Prompt `"Explain the concept of machine learning to a 10-year-old."`

`model`: `gpt-5.6-luna` and `gpt-5.4-mini`.

Compare the outputs from each model. 

You can evaluate the differences in:
* Clarity of explanation.
* Simplicity of language (suitable for a 10-year-old).
* Creativity in the response.
* Response time.

In [9]:
PROMPT_2 = "Explain the concept of machine learning to a 10-year-old."

def readability(text):
    words = re.findall(r"[A-Za-z']+", text)
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    return {"words": len(words),
            "sentences": len(sentences),
            "words_per_sentence": round(len(words) / max(len(sentences), 1), 1),
            "avg_word_length": round(sum(len(w) for w in words) / max(len(words), 1), 2)}

for model in (LUNA, MINI):
    resp, secs = ask(PROMPT_2, model=model)
    show(resp, secs, model)
    print("readability:", readability(texts(resp)[0]), "\n")

##### gpt-5.6-luna
--- answer 1 (finish_reason: stop) ---
Machine learning is a way to teach computers to make guesses or decisions by showing them lots of examples instead of giving them every rule.

Imagine teaching a computer to recognize cats:

1. You show it many pictures labeled “cat” and “not a cat.”
2. The computer notices patterns—like pointy ears, whiskers, and certain shapes.
3. Later, when it sees a new picture, it uses those patterns to guess whether there is a cat.

The computer doesn’t think exactly like a person. It finds patterns in information and uses them to make predictions.

Machine learning helps power things like:

- Voice assistants understanding speech
- Video and music recommendations
- Email spam filters
- Self-driving car features
- Photo apps recognizing faces or objects

It can sometimes make mistakes, just like people do, especially if it hasn’t seen enough good examples. 

tokens: input=19, output=177, total=196 | time: 2.70s

readability: {'words': 137

In [10]:
# One call is a noisy speed measurement, so I repeat it three times per model.
timings = {LUNA: [], MINI: []}
for _ in range(3):
    for model in timings:
        _, secs = ask(PROMPT_2, model=model)
        timings[model].append(secs)

for model, t in timings.items():
    print(f"{model:15s} runs: {[round(x, 2) for x in t]}  mean: {sum(t) / len(t):.2f}s")

gpt-5.6-luna    runs: [2.43, 2.05, 2.35]  mean: 2.28s
gpt-5.4-mini    runs: [2.33, 1.64, 2.06]  mean: 2.01s


**Comparison.** Both models should explain machine learning in a way a ten-year-old can follow, so the interesting differences are small ones: how many analogies each one uses, how long the answer is, how long the sentences are (the readability numbers above) and how fast it comes back. A bigger model usually writes a bit more and answers a bit slower, while the mini model is quicker and plainer. Response time from a single call can swing a lot with server load, which is why I timed three runs per model and look at the mean.

**From my run:** _(add one or two lines here after running: which answer was clearer, which was more creative, and the mean times)_

#### 3. Call the OpenAI API with different `temperature` values

Prompt: `"Write a two sentence story about a robot learning to cook."`

`temperature` values: `0`, `0.7` and `1.5`. Generate three responses at each temperature.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare the outputs.

Tip: With Chat Completions API you can use parameter `n=3` to generate three different responses to your prompt with one call.

In [11]:
PROMPT_3 = "Write a two sentence story about a robot learning to cook."

# Only the temperature changes. Model, prompt and output limit stay the same.
for temp in (0, 0.7, 1.5):
    resp, secs = ask(PROMPT_3, temperature=temp, n=3,
                     reasoning_effort="none", max_completion_tokens=150)
    show(resp, secs, f"temperature = {temp}")
    print("distinct answers:", len(set(texts(resp))), "out of 3\n")

##### temperature = 0
--- answer 1 (finish_reason: stop) ---
The robot carefully studied recipes, measuring flour and stirring soup with precise, whirring motions until the kitchen smelled like warm bread. When it finally served dinner, it proudly announced that it had learned the secret ingredient was not code, but patience. 

--- answer 2 (finish_reason: stop) ---
The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, practiced motions. When it finally served dinner, the robot’s sensors glowed with pride as everyone agreed that its first homemade soup was delicious. 

--- answer 3 (finish_reason: stop) ---
The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, practiced motions. When it finally served dinner, the robot’s sensors glowed with pride as everyone agreed that its first homemade soup was delicious. 

tokens: input=18, output=1

**Comparison.** Temperature controls how willing the model is to pick less likely words. At `0` it should almost always pick the most likely next word, so the three stories should be identical or nearly identical. At `0.7` I expect the same basic idea with different wording, which is the usual sweet spot for creative writing. At `1.5` the probabilities get flattened a lot, so the stories should differ much more and may contain odd word choices or sentences that stop making sense.

**From my run:** _(add one or two lines: how many distinct answers at each temperature, and whether 1.5 started to look strange)_

#### 4. Call the OpenAI API with different `max_output_tokens`/`max_completion_tokens` values.

Prompt: `"Explain the concept of machine learning in simple terms."`

`max_output_tokens` (Response API call)/`max_completion_tokens` (Chat Completions API call) values: `50`, `100` and `200`.

Use `gpt-5.4-mini` for this task.

Compare the length and detail of the responses.

In [12]:
PROMPT_4 = "Explain the concept of machine learning in simple terms."

# reasoning_effort="none" keeps all of the token budget for the visible answer.
# Without it, a reasoning model can spend the 50 tokens on hidden thinking and return nothing.
for limit in (50, 100, 200):
    resp, secs = ask(PROMPT_4, max_completion_tokens=limit, reasoning_effort="none")
    answer = texts(resp)[0]
    show(resp, secs, f"max_completion_tokens = {limit}")
    print(f"words in answer: {len(answer.split())}\n")

##### max_completion_tokens = 50
--- answer 1 (finish_reason: length) ---
Machine learning is a way to teach computers to learn from examples instead of being told every rule explicitly.

### Simple idea
Imagine you want a computer to recognize cats in photos.  
Instead of writing a long list like:
- ears are 

tokens: input=16, output=50, total=66 | time: 1.60s

words in answer: 42

##### max_completion_tokens = 100
--- answer 1 (finish_reason: length) ---
Machine learning is a way for computers to learn patterns from data instead of being told exact rules.

Simple example:
- If you show a computer many pictures of cats and dogs,
- it can learn what usually makes a cat look different from a dog,
- and later it can guess whether a new picture is a cat or a dog.

In other words:
- **Traditional programming:** humans write the rules
- **Machine learning:** humans give examples, and the computer figures out the rules 

tokens: input=16, output=100, total=116 | time: 1.82s

words in answer

**Comparison.** The limit does not make the model write a shorter, tidier answer. It just cuts the text off when the budget runs out, so at `50` tokens the answer should stop mid-sentence (`finish_reason: length`), at `100` it gets further into the explanation, and at `200` it may finish on its own (`finish_reason: stop`). A bigger budget gives room for examples and more detail. If I want a short but complete answer, the prompt has to ask for it (like "in two sentences"), because the limit is only a hard ceiling.

**From my run:** _(add a line: which limits ended with `length` and which ended with `stop`)_

#### 5. Call the OpenAI API with different `top_p` values.

Prompt: `"Write a list of 5 completely random and unusual hobbies that no one has ever heard of."`

`top_p` values: `0.3`, `0.7`, and `1.0`.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare response variation as you adjust `top_p`.

In [13]:
PROMPT_5 = "Write a list of 5 completely random and unusual hobbies that no one has ever heard of."

all_items = {}
for top_p in (0.3, 0.7, 1.0):
    resp, secs = ask(PROMPT_5, top_p=top_p, temperature=1.0, n=3,
                     reasoning_effort="none", max_completion_tokens=200)
    show(resp, secs, f"top_p = {top_p}")
    items = [i.lower() for t in texts(resp) for i in list_items(t)]
    all_items[top_p] = items
    print(f"hobbies listed: {len(items)}, different ones: {len(set(items))}\n")

##### top_p = 0.3
--- answer 1 (finish_reason: stop) ---
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow collecting** — tracking and sketching the shapes of shadows cast by clouds at different times of day.  
2. **Vintage key whispering** — collecting old keys and inventing fictional histories for each one.  
3. **Pebble balancing in impossible places** — trying to stack stones on surfaces that seem too uneven to work.  
4. **Map margin doodling** — decorating the blank borders of old maps with tiny imaginary creatures and symbols.  
5. **Rain sound cataloging** — recording and comparing the sound of rain on different surfaces like metal, leaves, glass, and dirt. 

--- answer 2 (finish_reason: stop) ---
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking and sketching the shapes that clouds cast on the ground at different times of day.  
2. **Vintage button cataloging** — collecting old buttons and organizing them by m

**Comparison.** `top_p` cuts the list of candidate words down to the smallest group whose probabilities add up to `top_p`. With `0.3` only the very likeliest words stay in play, so the three answers should look alike and reuse the same "unusual" hobbies. With `1.0` nothing is cut, so the model can reach for rarer ideas and the three lists should overlap less. The count of different hobbies printed above is a rough measure of that variety. I kept the temperature fixed at 1.0 so that `top_p` is the only thing that changes.

**From my run:** _(add a line with the "different ones" counts for 0.3, 0.7 and 1.0)_

#### 6. Call the OpenAI API with different `frequency_penalty` values.

Prompt: `"Write a list of funny one-liner jokes about cats."`

`frequency_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the repetition of joke formats changes as you increase the `frequency_penalty`.

In [14]:
PROMPT_6 = "Write a list of funny one-liner jokes about cats."

def opener_share(items, k=3):
    """Share of the list that starts with the most common first k words (1.0 = all jokes start the same)."""
    openers = [" ".join(re.findall(r"[a-z']+", i.lower())[:k]) for i in items]
    return Counter(openers).most_common(1)[0][1] / len(openers) if openers else 0.0

for fp in (0, 0.5, 1.0):
    resp, secs = ask(PROMPT_6, frequency_penalty=fp, n=3,
                     reasoning_effort="none", max_completion_tokens=300)
    show(resp, secs, f"frequency_penalty = {fp}")
    shares = [opener_share(list_items(t)) for t in texts(resp)]
    print("share of jokes with the same opening words:", [round(s, 2) for s in shares], "\n")

##### frequency_penalty = 0
--- answer 1 (finish_reason: length) ---
Sure — here are some funny cat one-liners:

1. My cat is not spoiled — she just has high purr-spectives.  
2. Cats don’t have owners; they have staff.  
3. I asked my cat for advice, but she just gave me a cold shoulder.  
4. My cat sleeps 18 hours a day because she’s working hard on her purr-sonal development.  
5. The cat joined the internet because she heard there would be a lot of cat-ching up.  
6. I’m convinced my cat thinks “No” is just a suggestion.  
7. My cat’s favorite hobby is ignoring me with commitment.  
8. Cats are great at interviews — they always nail the purr-formance review.  
9. My cat only comes when called… if I open a can first.  
10. I wanted a guard dog, but my cat said, “I’ll do it for the box.”  
11. My cat doesn’t do tricks; she does negotiations.  
12. I bought my cat a bed, and she preferred the cardboard box — she has impeccable taste in budget furniture.  
13. My cat’s favorite exercis

**Comparison.** The frequency penalty lowers the score of a word every time it has already been used, and the penalty grows with each repeat. For a list of cat jokes this should show up as less repetition of the same setup ("Why did the cat...") and of words like "cat", "purr" or "meow". At `0` I expect several jokes built on the same template. At `1.0` the model is pushed to vary the wording and the format. The "share of jokes with the same opening words" number above tracks this (lower means more varied). Very high penalties can also make the text slightly awkward because common words become expensive.

**From my run:** _(add a line: did the repeated "Why did the cat" style go down as the penalty went up?)_

#### 7. Call the OpenAI API with different `presence_penalty` values.

Prompt: `"Suggest ten names for a new line of eco-friendly cleaning products."`

`presence_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the diversity of generated names changes as you increase the `presence_penalty`.

In [15]:
PROMPT_7 = "Suggest ten names for a new line of eco-friendly cleaning products."

def clean_name(item):
    return re.split(r"\s[–—-]\s|:", item)[0].strip().lower()

for pp in (0, 0.5, 1.0):
    resp, secs = ask(PROMPT_7, presence_penalty=pp, n=3,
                     reasoning_effort="none", max_completion_tokens=300)
    show(resp, secs, f"presence_penalty = {pp}")
    names = [clean_name(i) for t in texts(resp) for i in list_items(t)]
    print(f"names listed: {len(names)}, different names: {len(set(names))}\n")

##### presence_penalty = 0
--- answer 1 (finish_reason: stop) ---
Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PureNest**
3. **EcoSpark**
4. **FreshLeaf Clean**
5. **NatureShine**
6. **BloomClean**
7. **EarthKind Essentials**
8. **VerdantSweep**
9. **CleanAura**
10. **Leaf & Lather**

If you want, I can also suggest names with a more **luxury**, **playful**, or **scientific/modern** feel. 

--- answer 2 (finish_reason: stop) ---
Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PureCycle**
3. **EarthSpark**
4. **EcoShine**
5. **NatureNest Clean**
6. **FreshRoot**
7. **VerdeVibe**
8. **CleanLeaf**
9. **BrightBiome**
10. **Bloom & Brite**

If you want, I can also suggest names that sound more premium, more playful, or more “all-natural.” 

--- answer 3 (finish_reason: stop) ---
Here are ten name ideas for an eco-friendly cleaning product line:

1. **GreenGlide**
2. **PureSprout**
3. **EarthSheen**
4.

**Comparison.** The presence penalty is a one-time penalty on any word that has already appeared at least once, no matter how often. That pushes the model toward new words and new topics, so with a higher value the names should use a wider range of words (less "Eco-" and "Green" in every name) and the three runs should share fewer names. At `0` the model happily reuses the same safe building blocks. The difference from the frequency penalty is that this one only cares whether a word was used, not how many times.

**From my run:** _(add a line with the "different names" counts for 0, 0.5 and 1.0)_

#### 8. Use the prompt "*Explain the importance of data preprocessing in machine learning.*" with the following system prompts:
1. "You are a university professor teaching a class of beginners."
2. "You are a technical expert explaining to a team of engineers."
3. "You are a comedian explaining this concept in a funny way."

Observe how the tone and style of the response change based on the system prompt.

In [16]:
PROMPT_8 = "Explain the importance of data preprocessing in machine learning."

system_prompts = {
    "1. University professor": "You are a university professor teaching a class of beginners.",
    "2. Technical expert": "You are a technical expert explaining to a team of engineers.",
    "3. Comedian": "You are a comedian explaining this concept in a funny way.",
}

for label, system in system_prompts.items():
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": PROMPT_8}]
    resp, secs = ask(messages=messages, reasoning_effort="none", max_completion_tokens=500)
    show(resp, secs, label)
    print("words:", len(texts(resp)[0].split()), "\n")

##### 1. University professor
--- answer 1 (finish_reason: stop) ---
Data preprocessing is one of the most important steps in machine learning because raw data is usually messy, incomplete, and inconsistent. Before a model can learn useful patterns, the data often needs to be cleaned and transformed.

### Why it matters

1. **Improves model accuracy**
   - Garbage in, garbage out: if the input data is noisy or incorrect, the model will learn poor patterns.
   - Cleaning data helps the model make better predictions.

2. **Handles missing values**
   - Real-world datasets often have blanks or incomplete entries.
   - Preprocessing can fill in missing values or remove incomplete records, depending on the situation.

3. **Removes noise and errors**
   - Data may contain duplicates, typos, outliers, or inconsistent formats.
   - Fixing these issues reduces confusion for the model.

4. **Makes features comparable**
   - Some features may have very different scales, like age and income.
   - 

**Observations.** The question is the same each time, so every difference comes from the system prompt. The professor version should be patient and step by step, with definitions and simple examples. The engineer version should be shorter and more technical (missing values, scaling, encoding, data leakage, train and test splits) and assume the reader already knows the basics. The comedian version should wrap the same facts in jokes and exaggerated comparisons. The facts stay roughly the same, while vocabulary, structure and tone change with the role the model is given.

**From my run:** _(add a line about the tone of each of the three answers)_

#### 9. Write a Zero-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!".

Use it in your API call.

In [17]:
REVIEW = "The product is amazing, and I love using it every day!"

zero_shot = (
    "Classify the sentiment of the following product review as Positive, Negative or Neutral.\n\n"
    f'Review: "{REVIEW}"'
)

SENTIMENT_SETTINGS = dict(temperature=0, reasoning_effort="none", max_completion_tokens=100)

resp, secs = ask(zero_shot, **SENTIMENT_SETTINGS)
show(resp, secs, "Zero-shot")

##### Zero-shot
--- answer 1 (finish_reason: stop) ---
Positive 

tokens: input=38, output=4, total=42 | time: 0.89s



#### 10. Write a Few-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!"

Use it in your API call.

Observe how the model's performance changes and compare the response to Zero-Shot output.

In [18]:
few_shot = (
    "Classify the sentiment of each product review as Positive, Negative or Neutral.\n\n"
    'Review: "Stopped working after two days. Total waste of money."\n'
    "Sentiment: Negative\n\n"
    'Review: "It does the job. Nothing special, nothing wrong."\n'
    "Sentiment: Neutral\n\n"
    'Review: "Great battery life and the screen is gorgeous."\n'
    "Sentiment: Positive\n\n"
    f'Review: "{REVIEW}"\n'
    "Sentiment:"
)

resp, secs = ask(few_shot, **SENTIMENT_SETTINGS)
show(resp, secs, "Few-shot")

##### Few-shot
--- answer 1 (finish_reason: stop) ---
Positive 

tokens: input=95, output=4, total=99 | time: 0.98s



In [19]:
# The given review is easy, so I also try two harder ones to see if the examples make a difference.
harder_reviews = [
    "Oh great, it broke on day one. Exactly what I wanted.",
    "The camera is fantastic but the battery barely lasts half a day.",
]

def zero_shot_prompt(r):
    return ("Classify the sentiment of the following product review as Positive, Negative or Neutral.\n\n"
            f'Review: "{r}"')

def few_shot_prompt(r):
    return few_shot.rsplit('Review: "', 1)[0] + f'Review: "{r}"\nSentiment:'

for r in harder_reviews:
    print("REVIEW:", r)
    for name, build in (("zero-shot", zero_shot_prompt), ("few-shot ", few_shot_prompt)):
        resp, _ = ask(build(r), **SENTIMENT_SETTINGS)
        print(f"  {name}: {texts(resp)[0]!r}")
    print()

REVIEW: Oh great, it broke on day one. Exactly what I wanted.
  zero-shot: 'Negative'
  few-shot : 'Negative'

REVIEW: The camera is fantastic but the battery barely lasts half a day.
  zero-shot: 'Negative'
  few-shot : 'Negative'



**Zero-shot vs few-shot.** For the original review both prompts should land on "Positive", because the review is very clear. The difference is mostly in the format. With zero-shot the model decides how to answer and may add a sentence of explanation, while with few-shot it copies the pattern from my examples and gives just the label, which is easier for a program to read. The examples also set what the labels mean, which matters more for tricky reviews like the sarcastic one or the mixed one above, where the zero-shot answer can differ from the few-shot answer.

**From my run:** _(add a line: did the zero-shot answer include extra text? did the harder reviews come out differently?)_

#### 11. Optimize input tokens.

Prompt: `"As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."`

Rewrite the prompt to use as few input tokens as possible while preserving its meaning and the requirement to answer in two simple sentences. The response should remain relevant, clear, and accurate.

Run the original and revised prompts using the same model and settings. Compare their input token counts and assess whether shortening the prompt affected the response quality.

You can use the API response’s usage information to check token counts.

In [20]:
ORIGINAL_11 = 'As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended.'

SHORT_11 = "Why might a sci-fi fan get a romantic comedy recommended? Answer in two simple sentences."

SETTINGS_11 = dict(temperature=0, reasoning_effort="none", max_completion_tokens=150)

runs_11 = {}
for label, prompt in (("original", ORIGINAL_11), ("shortened", SHORT_11)):
    resp, secs = ask(prompt, **SETTINGS_11)
    runs_11[label] = resp
    show(resp, secs, label)

a, b = runs_11["original"].usage.prompt_tokens, runs_11["shortened"].usage.prompt_tokens
print(f"input tokens: original = {a}, shortened = {b}, saved = {a - b} ({(a - b) / a:.0%})")

##### original
--- answer 1 (finish_reason: stop) ---
The recommendation system may have noticed that many people who like sci-fi also enjoy romantic comedies, so it suggests one based on similar users’ tastes. It could also be trying to show something new that fits a few of the user’s past preferences, even if it is a different genre. 

tokens: input=94, output=61, total=155 | time: 1.73s

##### shortened
--- answer 1 (finish_reason: stop) ---
A sci-fi fan might get a romantic comedy recommended because the recommendation system sees shared themes like adventure, humor, or strong character relationships. It may also be trying to suggest something different but still enjoyable based on the fan’s past viewing habits. 

tokens: input=24, output=51, total=75 | time: 1.39s

input tokens: original = 94, shortened = 24, saved = 70 (74%)


**Result.** The original prompt spends most of its tokens on background the model already knows (that streaming services recommend things based on viewing history). The short version keeps the two things that matter: the situation (a sci-fi fan gets a romantic comedy) and the format (two simple sentences). Check the token numbers printed above for the saving. Both answers should say roughly the same thing, such as the system testing something new, looking at what similar viewers watch, or using other data like time of day or a shared account. If the shortened answer is as clear and accurate as the original, then the cut tokens were not doing any work.

**From my run:** _(add a line: the two input token counts and whether the two answers were equally good)_

#### 12. Optimize output tokens.

Prompt: `"Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"`

Rewrite the prompt to request a concise, plausible explanation in 10 output tokens or fewer.

Run the original and revised prompts using the same model and settings. Compare their output token counts and check whether the shorter response still provides a meaningful explanation. Refine your prompt if necessary.

You can use the API response’s usage information to check token counts.

In [21]:
ORIGINAL_12 = "Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"

SETTINGS_12 = dict(temperature=0, reasoning_effort="none", max_completion_tokens=300)

resp, secs = ask(ORIGINAL_12, **SETTINGS_12)
show(resp, secs, "original")

revised_12 = [
    ORIGINAL_12 + " Answer in 8 words or fewer.",
    ORIGINAL_12 + " Reply with one short phrase, 6 words max, no intro.",
    "Movie app suggested a romcom despite my sci-fi habit. Why? Max 5 words, no intro.",
]

for i, prompt in enumerate(revised_12, 1):
    resp, secs = ask(prompt, **SETTINGS_12)
    show(resp, secs, f"revised prompt {i}")
    used = resp.usage.completion_tokens
    print("within 10 output tokens:", "YES" if used <= 10 else "no", f"({used} tokens)\n")

##### original
--- answer 1 (finish_reason: stop) ---
A movie app can recommend a romantic comedy even if you mostly watch sci-fi for a few common reasons:

- **One-off behavior**: You may have watched, searched for, or clicked on a rom-com once, and the app weighted that heavily.
- **Shared signals**: If you share an account or device, someone else’s viewing can affect recommendations.
- **Broad similarity**: The app may think the rom-com is similar to sci-fi based on actors, themes, pacing, or audience overlap.
- **Popularity bias**: Highly popular titles often get recommended to many users, regardless of taste.
- **Cold-start / limited data**: If the app doesn’t have enough viewing history, it may make more generic suggestions.
- **Contextual factors**: Time of day, recent trends, or what’s new on the platform can influence recommendations.
- **Algorithm experimentation**: Some apps intentionally mix in “exploration” picks to see if you’ll branch out.

If you want, I can also help y

**Result.** The trick is to ask for a length in words, because the model counts words much more reliably than tokens, and then check the real token count in `usage`. Short answers cost fewer output tokens, but there is a limit: at five or six words the answer is usually something like "Trying new genres to keep you engaged", which is still a plausible reason, whereas going much shorter drops the explanation altogether. The best revised prompt is the shortest one that stays at 10 tokens or fewer and still gives an actual reason.

**From my run:** _(add a line: which revised prompt you picked, its output token count, and the answer it gave)_

#### 13. Expand the given prompt accordingly to achieve the requested output from the list below.

Prompt: `"Explain the importance of exercise." `

Expected output (from three separate prompts):
1. One sentence long
2. Paragraph format
3. Bullet points

In [22]:
BASE_13 = "Explain the importance of exercise."

prompts_13 = {
    "1. One sentence": BASE_13 + " Answer in exactly one sentence.",
    "2. Paragraph": BASE_13 + " Write it as a single paragraph of about 5 sentences, with no bullet points or headings.",
    "3. Bullet points": BASE_13 + " Answer only with 5 bullet points, each under 15 words, with no introduction or closing line.",
}

run_prompt_set(prompts_13)

PROMPT: Explain the importance of exercise. Answer in exactly one sentence.
##### 1. One sentence
--- answer 1 (finish_reason: stop) ---
Exercise is important because it strengthens the body, improves heart and mental health, boosts energy and mood, and helps prevent many chronic diseases. 

tokens: input=18, output=30, total=48 | time: 1.31s

PROMPT: Explain the importance of exercise. Write it as a single paragraph of about 5 sentences, with no bullet points or headings.
##### 2. Paragraph
--- answer 1 (finish_reason: stop) ---
Exercise is important because it helps keep the body healthy and strong. It improves heart health, builds muscle, and supports a healthy weight. Regular physical activity can also boost mood, reduce stress, and improve sleep. In addition, exercise helps prevent many diseases such as diabetes and high blood pressure. Overall, staying active is one of the best ways to maintain both physical and mental well-being. 

tokens: input=31, output=80, total=111 | time: 

#### 14. Expand this prompt to get more creative responses.

Prompt: `"Explain Artificial Intelligence."`

Expected output (from three separate prompts):
1. Metaphor
2. Story
3. Funny

In [23]:
BASE_14 = "Explain Artificial Intelligence."

prompts_14 = {
    "1. Metaphor": BASE_14 + " Use one extended metaphor and let it carry the whole explanation, without giving a literal definition.",
    "2. Story": BASE_14 + " Do it by telling a short story of about 150 words with a character and a small problem that gets solved.",
    "3. Funny": BASE_14 + " Make it genuinely funny, with jokes and silly comparisons, but keep the explanation correct and under 120 words.",
}

run_prompt_set(prompts_14)

PROMPT: Explain Artificial Intelligence. Use one extended metaphor and let it carry the whole explanation, without giving a literal definition.
##### 1. Metaphor
--- answer 1 (finish_reason: stop) ---
Artificial Intelligence is a vast library that does not merely store books, but learns how to rearrange them into new rooms.

At first, the library is empty of understanding. Then people begin to bring in shelves of examples: pictures, sentences, sounds, decisions, mistakes, victories. The library studies the patterns in how these books are written, how chapters tend to follow one another, which pages echo which others, where the common paths are, where the rare turns appear. It does not read like a scholar seeking a single truth; it reads like a cartographer tracing the shape of a landscape from countless footprints.

Over time, the library becomes skilled at prediction. If you open one book to a certain page, it can often guess what kind of page comes next. If you place a fragment of a 

#### 15. Expand this prompt for different audiences.

Prompt: `"Explain climate change."`

Expected audience (from three separate prompts):
1. Kids
2. General public
3. Politicians

In [24]:
BASE_15 = "Explain climate change."

prompts_15 = {
    "1. Kids": BASE_15 + " The audience is children around 8 years old. Use very simple words, a friendly tone and one everyday comparison. Keep it under 100 words.",
    "2. General public": BASE_15 + " The audience is the general public with no science background. Be clear and balanced, use plain language, and keep it to about 150 words.",
    "3. Politicians": BASE_15 + " The audience is politicians who must make policy decisions. Focus on causes, economic and security impacts, policy options and trade-offs, and keep it to about 150 words.",
}

run_prompt_set(prompts_15)

PROMPT: Explain climate change. The audience is children around 8 years old. Use very simple words, a friendly tone and one everyday comparison. Keep it under 100 words.
##### 1. Kids
--- answer 1 (finish_reason: stop) ---
Climate change means the Earth is getting warmer little by little. This happens because people burn fuels like gas and coal, which sends gases into the air. These gases act like a thick blanket around the Earth, trapping heat. It’s a bit like wearing too many blankets on a hot day. A warmer Earth can cause stronger storms, hotter days, and less ice in some places. We can help by saving energy, planting trees, and using less car travel. 

tokens: input=40, output=95, total=135 | time: 1.96s

PROMPT: Explain climate change. The audience is the general public with no science background. Be clear and balanced, use plain language, and keep it to about 150 words.
##### 2. General public
--- answer 1 (finish_reason: stop) ---
Climate change means long-term changes in Earth’

#### 16. Expand this prompt accordingly to instructions below.

Prompt: `"Describe a sunset."`

Expected result (from three separate prompts):

1. Add more details to the prompt, e.g., on what details should it focus on.
2. Constrain the prompt by forbidding it to use words 'sun' and 'light'.
3. Change the output in the style of Shakespeare.

In [25]:
BASE_16 = "Describe a sunset."

prompts_16 = {
    "1. More detail": BASE_16 + " Focus on the changing colours across the sky, the sounds nearby, the temperature of the air and the mood of the place. Set it over the sea and use about 120 words.",
    "2. Forbidden words": BASE_16 + " Use about 100 words. Do not use the words 'sun' or 'light', or any word that contains them (for example sunset, sunlight, sunny, lighten, lightly).",
    "3. Shakespeare": BASE_16 + " Write it in the style of Shakespeare, in an Elizabethan voice with iambic rhythm, about 10 to 12 lines.",
}

run_prompt_set(prompts_16)

# Check the constraint from prompt 2 with code instead of by eye.
resp, _ = ask(prompts_16["2. Forbidden words"], reasoning_effort="none", max_completion_tokens=400)
banned = re.findall(r"\w*(?:sun|light)\w*", texts(resp)[0], flags=re.I)
print("Forbidden words found in a fresh run of prompt 2:", banned if banned else "none")

PROMPT: Describe a sunset. Focus on the changing colours across the sky, the sounds nearby, the temperature of the air and the mood of the place. Set it over the sea and use about 120 words.
##### 1. More detail
--- answer 1 (finish_reason: stop) ---
The sun lowers over the sea, spilling a slow wash of colour across the sky. Near the horizon, the light burns gold and tangerine, then softens into bands of rose, lavender, and pale blue higher above. The water catches each shade and breaks it into shimmering paths that tremble with the small waves. Nearby, gulls call in sharp, echoing cries, and the hush of the tide folding against the shore gives the scene a gentle rhythm. The air is cooler now, carrying a faint saltiness that brushes the skin with a clean, evening chill. Everything feels calm and unhurried, as if the world has paused to breathe, holding a quiet, wistful beauty before night arrives. 

tokens: input=47, output=147, total=194 | time: 1.61s

PROMPT: Describe a sunset. Use a

#### 17. Write your observations from tasks 13-16, e.g., how different prompts influence complexity, tone, style and focus of the responses. 

#### My observations for tasks 13-16

**Format and length (task 13).** The topic is identical in all three answers. What changes is the amount of content: one sentence keeps only the most general benefit, the paragraph adds a few reasons and links them together, and the bullets split the same ideas into separate scannable points. Saying exactly how many sentences or bullets I want, and what to leave out ("no introduction"), made the result match much better than the bare prompt would.

**Creative style (task 14).** A metaphor, a story and a joke-based version explain the same subject in very different ways. The metaphor trades accuracy for a picture the reader can remember, the story spreads the idea over a character and a plot, and the funny version leans on exaggeration and wordplay. Each is easier to remember than a plain definition but loses some precision, so for anything that must be exact the plain version still wins.

**Audience (task 15).** The audience changes vocabulary, depth and focus. For kids the answer uses short sentences and an everyday comparison. For the general public it explains causes and effects without jargon. For politicians the focus moves from "what is happening" to costs, risks, trade-offs and policy options. The facts hardly change, while the choice of what to emphasize does.

**Detail, constraints and style (task 16).** Telling the model what to focus on (colours, sound, temperature, mood) gave a richer and more specific description than "describe a sunset" would. The forbidden-words constraint forced more inventive wording (the check in the code cell shows whether it held, and models sometimes slip on words like "sunset" even when told not to). The Shakespeare version shows how strongly a style instruction changes rhythm, word choice and sentence structure.

**Overall.** The more precisely the prompt states the format, the length, the audience, the focus and any hard constraints, the less the model has to guess and the closer the answer is to what I wanted. Vague prompts get a generic answer.

**From my run:** _(add anything specific you noticed, such as whether a constraint was broken)_

#### 18. Generate a Python code with API call.

Write a prompt that generates a Python function to calculate the factorial of a number. Once you have written the prompt, use it to call the API and generate the code. Test the generated code to ensure it calculates the factorial correctly for various inputs.

In [26]:
PROMPT_18 = (
    "Write a Python function named factorial(n) that returns n! for a non-negative integer n. "
    "Use a loop, not recursion. Raise ValueError if n is negative and TypeError if n is not an integer. "
    "Include a one-line docstring. "
    "Reply with only the code in a single ```python code block, with no explanation and no example calls."
)

resp, secs = ask(PROMPT_18, reasoning_effort="none", temperature=0, max_completion_tokens=400)
show(resp, secs, "Generated code")
generated_18 = extract_code(texts(resp)[0])

##### Generated code
--- answer 1 (finish_reason: stop) ---
```python
def factorial(n):
    """Return n! for a non-negative integer n."""
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    if n < 0:
        raise ValueError("n must be non-negative")
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
``` 

tokens: input=77, output=88, total=165 | time: 1.85s



In [27]:
namespace = {}
exec(generated_18, namespace)
factorial = namespace["factorial"]

for n in [0, 1, 2, 5, 10, 20]:
    got, want = factorial(n), math.factorial(n)
    print(f"factorial({n}) = {got}  {'PASS' if got == want else 'FAIL, expected ' + str(want)}")

for bad, expected in [(-3, ValueError), (2.5, TypeError)]:
    try:
        factorial(bad)
        print(f"factorial({bad!r}) did not raise anything  FAIL")
    except expected:
        print(f"factorial({bad!r}) raised {expected.__name__}  PASS")
    except Exception as e:
        print(f"factorial({bad!r}) raised {type(e).__name__} instead of {expected.__name__}  FAIL")

factorial(0) = 1  PASS
factorial(1) = 1  PASS
factorial(2) = 2  PASS
factorial(5) = 120  PASS
factorial(10) = 3628800  PASS
factorial(20) = 2432902008176640000  PASS
factorial(-3) raised ValueError  PASS
factorial(2.5) raised TypeError  PASS


**Why the prompt works.** It names the function and its argument, says how to implement it (loop, no recursion), says what the edge cases should do (negative numbers, wrong types), and asks for code only in one fenced block so I can extract it automatically. I tested the result against `math.factorial` for several inputs, including 0 and 1, and checked the error cases. The tests above print PASS or FAIL for each one.

#### 19. Provide the following buggy Python code to the API and ask it to fix the errors:

```
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
```

Check if the fixed code is working.

In [28]:
BUGGY = """def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
"""

try:
    compile(BUGGY, "buggy.py", "exec")
except SyntaxError as e:
    print("Original code fails:", e.msg, f"(line {e.lineno})\n")

PROMPT_19 = (
    "Fix the errors in this Python code. Reply with the corrected code in one ```python code block, "
    "followed by one short line saying what you changed.\n\n```python\n" + BUGGY + "```"
)

resp, secs = ask(PROMPT_19, reasoning_effort="none", temperature=0, max_completion_tokens=400)
show(resp, secs, "Fixed by the model")
fixed_19 = extract_code(texts(resp)[0])

Original code fails: expected ':' (line 1)

##### Fixed by the model
--- answer 1 (finish_reason: stop) ---
```python
def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))
```

Fixed the missing colons and closing parenthesis. 

tokens: input=62, output=40, total=102 | time: 1.08s



In [29]:
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    exec(fixed_19, {})

printed = buffer.getvalue().strip()
print("Output of the fixed code:", printed)
print("Correct (expected 15):", printed == "15")

Output of the fixed code: 15
Correct (expected 15): True


**Result.** There were two syntax errors: the missing colon after the function definition line and the missing closing bracket in the `print` call. The `compile` check above shows Python failing on the original, and the second cell runs the model's version and compares the printed value with the expected `15`.

#### 20. Summarize this text to 2-3 sentences by using API call.

"Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."

[<p align="right"> <cite> Source: What is generative AI? </cite></p>](https://www.ibm.com/think/topics/generative-ai)

In [30]:
TEXT_20 = 'Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.\n\nAt a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.\n\nGenerative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:\n\nVariational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.\n\nDiffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.\n\nTransformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney.'

PROMPT_20 = "Summarize the following text in 2-3 sentences.\n\nText:\n" + TEXT_20

resp, secs = ask(PROMPT_20, reasoning_effort="none", temperature=0, max_completion_tokens=300)
show(resp, secs, "Summary")

summary = texts(resp)[0]
n_sentences = len([s for s in re.split(r"(?<=[.!?])\s+", summary) if s])
print("sentences in summary:", n_sentences, "| words:", len(summary.split()), "vs", len(TEXT_20.split()), "in the original")

##### Summary
--- answer 1 (finish_reason: stop) ---
Generative AI uses deep learning models to create new content like text, images, video, or audio based on a prompt, by learning patterns from training data and generating similar but original outputs. Key model types behind modern generative AI include VAEs, diffusion models, and especially transformers, which power many popular tools such as ChatGPT, GPT-4, Copilot, BERT, Bard, and Midjourney. 

tokens: input=309, output=87, total=396 | time: 1.11s

sentences in summary: 2 | words: 60 vs 222 in the original


**Check.** A good summary here should say that generative AI means deep learning models that create new content, mention the three model types (VAEs, diffusion models, transformers) and say that transformers sit behind most of the well-known tools. The sentence count printed above shows whether the 2-3 sentence limit was kept.

#### 21. Translate the following idioms into a language of your choice, ensuring the meaning is preserved (not just word-for-word translation):

1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise."

Try to improve the response as much as possible and evaluate the accuracy of the final output.

In [31]:
TARGET_LANGUAGE = "Spanish"   # change this to any language you can judge yourself

IDIOMS = [
    "It's raining cats and dogs.",
    "Break a leg.",
    "Kill two birds with one stone.",
    "Spill the beans.",
    "A blessing in disguise.",
]
idiom_list = "\n".join(f"{i}. {s}" for i, s in enumerate(IDIOMS, 1))

# Version 1: the plain request
naive = f"Translate these idioms into {TARGET_LANGUAGE}:\n{idiom_list}"
resp, secs = ask(naive, reasoning_effort="none", temperature=0, max_completion_tokens=500)
show(resp, secs, "Version 1: plain request")
v1 = texts(resp)[0]

##### Version 1: plain request
--- answer 1 (finish_reason: stop) ---
Here are natural Spanish equivalents for those idioms:

1. **It's raining cats and dogs.** → **Está lloviendo a cántaros.**
2. **Break a leg.** → **¡Mucha suerte!** / **¡Mucha mierda!** (common in theater)
3. **Kill two birds with one stone.** → **Matar dos pájaros de un tiro.**
4. **Spill the beans.** → **Soltar la sopa.** / **Revelar el secreto.**
5. **A blessing in disguise.** → **No hay mal que por bien no venga.**

If you want, I can also give you the **literal translations** or **more idiomatic options by country**. 

tokens: input=49, output=159, total=208 | time: 1.44s



In [32]:
# Version 2: tell the model what "good" means
improved = (
    f"Translate these English idioms into {TARGET_LANGUAGE}. Do not translate word for word. "
    f"For each one, use the idiom a native {TARGET_LANGUAGE} speaker would really say in the same situation. "
    "If there is no equivalent idiom, use a natural phrase with the same meaning and mark it with (no direct idiom). "
    "Give the answer as a numbered list in this format:\n"
    f"N. English idiom -> {TARGET_LANGUAGE} idiom | what it means in plain English\n\n"
    + idiom_list
)
resp, secs = ask(improved, reasoning_effort="none", temperature=0, max_completion_tokens=600)
show(resp, secs, "Version 2: meaning-first instructions")
v2 = texts(resp)[0]

##### Version 2: meaning-first instructions
--- answer 1 (finish_reason: stop) ---
1. It's raining cats and dogs. -> Está lloviendo a cántaros. | It’s raining very heavily.
2. Break a leg. -> ¡Mucha mierda! | Good luck, especially before a performance.
3. Kill two birds with one stone. -> Matar dos pájaros de un tiro. | Achieve two things with one action.
4. Spill the beans. -> Soltar la sopa. | Reveal a secret.
5. A blessing in disguise. -> No hay mal que por bien no venga. | Something that seems bad at first but turns out to be good. 

tokens: input=131, output=126, total=257 | time: 1.81s



In [33]:
# Version 3: ask the model to review its own translations
review = (
    f"Here are {TARGET_LANGUAGE} translations of English idioms:\n\n{v2}\n\n"
    f"Act as a careful native {TARGET_LANGUAGE} speaker. For each line say whether it is a real, natural expression "
    "(not a literal translation) and whether it is used in everyday speech. Fix any line that is weak, "
    "then give the final corrected list in the same numbered format."
)
resp, secs = ask(review, reasoning_effort="none", temperature=0, max_completion_tokens=900)
show(resp, secs, "Version 3: self-review")

##### Version 3: self-review
--- answer 1 (finish_reason: stop) ---
Here’s a careful review as a native Spanish speaker:

1. **Está lloviendo a cántaros.** — **Yes, real and natural.** Very common in everyday speech.
2. **¡Mucha mierda!** — **Yes, real and natural.** It’s the standard theatrical way to wish good luck, though it’s mainly used in performance/theater contexts, not in general everyday situations.
3. **Matar dos pájaros de un tiro.** — **Yes, real and natural.** Very common and widely used.
4. **Soltar la sopa.** — **Yes, real and natural.** It’s understood and used in everyday speech, though **“soltar el secreto”** or **“contar el secreto”** can also sound more direct depending on context.
5. **No hay mal que por bien no venga.** — **Yes, real and natural.** Very common in everyday speech.

### Final corrected list
1. It’s raining cats and dogs. -> **Está lloviendo a cántaros.** | It’s raining very heavily.  
2. Break a leg. -> **¡Mucha mierda!** | Good luck, especially be

**Evaluation.** The plain request is where literal translations tend to appear (for example, translating "raining cats and dogs" with animals in it). Spelling out the goal in version 2 pushes the model toward real idioms, and version 3 adds a review pass that catches lines that are correct but unnatural. For Spanish, these are the answers I would accept as natural:

| English idiom | Natural Spanish equivalents |
|---|---|
| It's raining cats and dogs. | *Llueve a cántaros* / *Está lloviendo a mares* |
| Break a leg. | *¡Mucha mierda!* (theatre tradition), or simply *¡Mucha suerte!* |
| Kill two birds with one stone. | *Matar dos pájaros de un tiro* |
| Spill the beans. | *Soltar la sopa* / *Irse de la lengua* |
| A blessing in disguise. | *No hay mal que por bien no venga* |

The usual mistakes to look for are a literal "Rómpete una pierna" for "Break a leg" (it exists but is a borrowed phrase, not the native idiom) and a word-for-word version of "spill the beans". If you picked another language, replace this table with your own check.

**From my run:** _(add a line: which lines of version 1 were wrong or literal, and whether version 3 fixed them)_

#### 22. Generate a paragraph about a cat without using the letter 'e'/'E'.

Write a prompt that forces the AI to produce a three-sentence story about a cat while strictly forbidding it to use letter 'e'.

Use `gpt-5.4-mini` for this task.

Explain why your prompt is effective.

In [34]:
PROMPT_22 = (
    "Write a story about a cat in exactly three sentences. "
    "STRICT RULE: the letter 'e' (lower or upper case) must not appear in any word, not even once. "
    "Check every word letter by letter before you answer. "
    "Words that contain 'e' and must be avoided include: the, he, she, we, me, be, her, their, there, then, when, were, "
    "have, like, one, some, came, made, every, over, under, after, never. "
    "Words that are safe: a, an, this, that, and, but, with, his, our, cat, nap, sat, ran, at, on, in, by, as, was, had, "
    "did, from, it, its, I, you, not, so, up, so, too, all, if, or, who, how, what, big, small, warm, soft, gold, black. "
    "Reply with the story only."
)

def words_with_e(text):
    return [w for w in re.findall(r"\w+", text) if "e" in w.lower()]

attempt_prompt = PROMPT_22
for attempt in range(1, 5):
    resp, secs = ask(attempt_prompt, model=MINI, max_completion_tokens=2000)
    story = texts(resp)[0]
    bad = words_with_e(story)
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", story) if s]
    print(f"--- attempt {attempt} ---\n{story}\n")
    print(f"sentences: {len(sentences)} | words containing 'e': {bad if bad else 'none'} | output tokens: {resp.usage.completion_tokens}\n")
    if not bad and len(sentences) == 3:
        print("Constraint satisfied.")
        break
    attempt_prompt = PROMPT_22 + f"\n\nYour previous attempt used these words with an 'e': {bad}. Write a new story that avoids them."

--- attempt 1 ---
A cat sat on a warm mat, and it had a soft nap.  
It did not run or hop; it just sat in a sun spot.  
I was glad, for that cat was a calm and small joy.

sentences: 3 | words containing 'e': none | output tokens: 55

Constraint satisfied.


**Why the prompt is effective.** Models do not see letters, only tokens, so a plain "do not use the letter e" is easy to break by accident. The prompt helps in four ways. It states the rule as strict and without exceptions. It tells the model to check word by word. It names the words that most often cause a slip ("the", "he", "she", "were") and gives safe replacements, so the model does not have to find them while writing. And the loop in the code checks the result programmatically and, if a word with an "e" slips through, feeds that list back for the next try. I left the reasoning setting at its default on purpose for this task, because the model checking its own words before answering is what makes it reliable.

**From my run:** _(add a line: how many attempts were needed)_

#### 23. Extract JSON Data from the input text in two different ways.

Input Text: "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

Extract the information from the text above using two different methods:

1. Write a prompt asking the AI to return a JSON object with the fields `destination`, `start_date`, and `stay_duration`.

2. Instead of putting rules in the prompt, provide a specific JSON schema that defines the field names and their required data types `destination` (the first city), `start_date` (formatted as YYYY-MM-DD), and `stay_duration` (the number of nights as an integer).

Which method yields a more reliable result for a computer program to process, and why?

In [35]:
INPUT_23 = "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

def parse_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)   # strip a code fence if the model added one
    return json.loads(text)

def trip_problems(d):
    problems = []
    if set(d) != {"destination", "start_date", "stay_duration"}:
        problems.append(f"wrong keys {sorted(d)}")
    if not isinstance(d.get("destination"), str):
        problems.append("destination is not a string")
    if not (isinstance(d.get("start_date"), str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", d["start_date"])):
        problems.append(f"start_date not YYYY-MM-DD: {d.get('start_date')!r}")
    sd = d.get("stay_duration")
    if not (isinstance(sd, int) and not isinstance(sd, bool)):
        problems.append(f"stay_duration is not an integer: {sd!r}")
    return problems

def check_runs(label, texts_list):
    ok = 0
    for t in texts_list:
        try:
            problems = trip_problems(parse_json(t))
        except json.JSONDecodeError as e:
            problems = [f"not valid JSON ({e.msg})"]
        ok += not problems
        print("   ", "OK " if not problems else "BAD", t.replace("\n", " ")[:110], "" if not problems else f"<- {problems}")
    print(f"{label}: {ok} of {len(texts_list)} runs usable by a program\n")

YEAR_NOTE = "The text does not give a year, so use 2026."

In [36]:
# Method 1: the rules are written in the prompt
method_1 = (
    "Extract the trip details from the text and return a JSON object with the fields "
    "destination, start_date and stay_duration. "
    "destination is the first city, start_date is formatted as YYYY-MM-DD, stay_duration is the number of nights as an integer. "
    + YEAR_NOTE + "\n\nText: " + INPUT_23
)
resp, secs = ask(method_1, n=5, reasoning_effort="none", temperature=1.0, max_completion_tokens=200)
check_runs("Method 1 (prompt only)", texts(resp))

    OK  {"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5} 
    OK  {"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5} 
    OK  {"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5} 
    OK  {   "destination": "Tokyo",   "start_date": "2026-06-12",   "stay_duration": 5 } 
    OK  {"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5} 
Method 1 (prompt only): 5 of 5 runs usable by a program



In [37]:
# Method 2: a JSON schema, passed as structured output
trip_schema = {
    "type": "object",
    "properties": {
        "destination": {"type": "string", "description": "The first city of the trip"},
        "start_date": {"type": "string", "description": "Start date formatted as YYYY-MM-DD"},
        "stay_duration": {"type": "integer", "description": "Number of nights"},
    },
    "required": ["destination", "start_date", "stay_duration"],
    "additionalProperties": False,
}

method_2 = "Extract the trip details from the text. " + YEAR_NOTE + "\n\nText: " + INPUT_23
resp, secs = ask(
    method_2, n=5, reasoning_effort="none", temperature=1.0, max_completion_tokens=200,
    response_format={"type": "json_schema",
                     "json_schema": {"name": "trip", "strict": True, "schema": trip_schema}},
)
check_runs("Method 2 (JSON schema)", texts(resp))

    OK  {"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5} 
    OK  {"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5} 
    OK  {"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5} 
    OK  {"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5} 
    OK  {"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5} 
Method 2 (JSON schema): 5 of 5 runs usable by a program



**Which method is more reliable, and why.** The schema method. With rules in the prompt, the model usually follows them, but nothing guarantees it: it may add a sentence before the JSON, wrap it in a code fence, write the date as "June 12" or return the number of nights as the text "5 nights". Any of those breaks `json.loads` or a later type check in a program. With a schema (structured output), the API forces the answer to match: the field names, the types and the required fields are guaranteed, so the program can parse it without defensive code. The runs above count how many of five answers each method got right. One thing a schema cannot do is invent missing facts: the text has no year, so I told the model to use 2026, and the date format `YYYY-MM-DD` is only enforced as "a string" by the schema, with the description as guidance.

**From my run:** _(add a line: how many of the 5 runs were usable for each method)_

#### 24. Create a structured table comparing different programming languages.

Write a prompt that generates a table in this format (example below) with accurate information.

| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data Science, AI       | Easy to learn, versatile       | Slower execution speed       |
| Java        | Enterprise Applications| Platform-independent, robust   | Verbose syntax               |
| JavaScript  | Web Development        | Fast, widely supported         | Limited for backend tasks    |

**Tip**: To enhance readability, you can copy and paste the generated table into a Markdown cell.

In [38]:
PROMPT_24 = (
    "Create a comparison table of six programming languages: Python, Java, JavaScript, C++, Go and Rust. "
    "Use exactly this Markdown format, with these four column headers, one row per language, "
    "and no more than 8 words in each cell. Only include widely accepted, factual points. "
    "Reply with the table only.\n\n"
    "| Language    | Use Case               | Advantages                     | Disadvantages                |\n"
    "|-------------|------------------------|---------------------------------|------------------------------|\n"
    "| Python      | Data Science, AI       | Easy to learn, versatile       | Slower execution speed       |\n"
    "| Java        | Enterprise Applications| Platform-independent, robust   | Verbose syntax               |\n"
    "| JavaScript  | Web Development        | Fast, widely supported         | Limited for backend tasks    |\n"
)

resp, secs = ask(PROMPT_24, reasoning_effort="none", temperature=0, max_completion_tokens=700)
table = texts(resp)[0]
print(table, "\n")

rows = [l for l in table.splitlines() if l.strip().startswith("|")]
cells_per_row = {len([c for c in r.strip().strip("|").split("|")]) for r in rows}
print("rows (incl. header and divider):", len(rows), "| columns per row:", cells_per_row)
print("tokens:", resp.usage.prompt_tokens, "in,", resp.usage.completion_tokens, "out")
display(Markdown(table))

| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data science, scripting | Readable, extensive libraries   | Slower than compiled languages |
| Java        | Enterprise applications | Portable, mature ecosystem       | Verbose syntax                |
| JavaScript  | Web development        | Ubiquitous in browsers          | Dynamic typing pitfalls       |
| C++        | Systems programming    | High performance, fine control   | Complex syntax, manual memory |
| Go          | Cloud services, tooling| Simple concurrency, fast builds  | Smaller ecosystem             |
| Rust         | Systems programming    | Memory safety, high performance | Steeper learning curve        | 

rows (incl. header and divider): 8 | columns per row: {4}
tokens: 158 in, 153 out


| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data science, scripting | Readable, extensive libraries   | Slower than compiled languages |
| Java        | Enterprise applications | Portable, mature ecosystem       | Verbose syntax                |
| JavaScript  | Web development        | Ubiquitous in browsers          | Dynamic typing pitfalls       |
| C++        | Systems programming    | High performance, fine control   | Complex syntax, manual memory |
| Go          | Cloud services, tooling| Simple concurrency, fast builds  | Smaller ecosystem             |
| Rust         | Systems programming    | Memory safety, high performance | Steeper learning curve        |

**Note.** I gave the model the exact header and three example rows, named the languages and capped the cell length. That keeps the columns identical to the example and the table compact, and the check above confirms every row has four columns. For accuracy I asked for widely accepted facts only, and I read through the rendered table myself (for example, Rust's main drawback is the steep learning curve and Go's is its limited feature set, not "slow speed").

#### 25. Generate a quiz about machine learning basics.

Look at the example below and write a prompt that generates a quiz with identical formatting. Ensure the title formatting, list markers (a, b, c, d), and the correct answer indicator (*) are accurately replicated.

**Machine learning basics quiz**

1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?  
a) Sorting files manually on a computer  
b) Email spam filtering *  
c) Writing code by hand  
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?  
a) When the model performs well on new, unseen data  
b) When the model is too simple and underperforms  
c) When the model learns the training data too well, including noise, reducing generalization *  
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?  
a) Increased hardware costs  
b) Bias in training data leading to unfair predictions *  
c) The inability to process large datasets  
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Manual learning *

In [39]:
quiz_items = [
    ("Which of the following best describes supervised learning?",
     ["The model learns from unlabeled data to find hidden patterns",
      "The model learns from labeled data to make predictions *",
      "The model is programmed with explicit rules",
      "The model randomly guesses outputs without training"]),
    ("Which of these is a common application of machine learning in everyday life?",
     ["Sorting files manually on a computer", "Email spam filtering *",
      "Writing code by hand", "Using a calculator for arithmetic"]),
    ("What is overfitting in machine learning?",
     ["When the model performs well on new, unseen data",
      "When the model is too simple and underperforms",
      "When the model learns the training data too well, including noise, reducing generalization *",
      "When the model runs faster than expected"]),
    ("Which ethical concern is commonly associated with machine learning algorithms?",
     ["Increased hardware costs", "Bias in training data leading to unfair predictions *",
      "The inability to process large datasets", "Lack of available programming languages"]),
    ("Which of the following is NOT a type of machine learning?",
     ["Supervised learning", "Reinforcement learning", "Unsupervised learning", "Manual learning *"]),
]

def format_quiz(title, items):
    blocks = []
    for i, (question, options) in enumerate(items, 1):
        lines = [f"{i}. {question}"] + [f"{'abcd'[j]}) {o}" for j, o in enumerate(options)]
        blocks.append("  \n".join(lines))          # two trailing spaces = Markdown line break
    return f"**{title}**\n\n" + "\n\n".join(blocks)

QUIZ_EXAMPLE = format_quiz("Machine learning basics quiz", quiz_items)

PROMPT_25 = (
    "Write a 5-question multiple-choice quiz about machine learning basics for beginners. "
    "Use exactly the same formatting as the example below:\n"
    "- the title in bold on the first line: **Machine learning basics quiz**\n"
    "- questions numbered 1. to 5.\n"
    "- four options per question, labelled a) b) c) d), each on its own line\n"
    "- end the question line and the first three option lines with two spaces so Markdown keeps the line breaks\n"
    "- mark the one correct option by adding ' *' at the end of that option\n"
    "- exactly one correct option per question, and vary which letter is correct\n"
    "Do not reuse the example questions. Reply with the quiz only, with no extra text.\n\n"
    "Example:\n\n" + QUIZ_EXAMPLE
)

resp, secs = ask(PROMPT_25, reasoning_effort="none", temperature=0.7, max_completion_tokens=900)
quiz = texts(resp)[0]
print(quiz)

**Machine learning basics quiz**

1. What is a training dataset used for in machine learning?  
a) To test the model after it is built  
b) To store the final predictions  
c) To teach the model patterns from examples *  
d) To remove all errors from the data

2. Which task is an example of classification?  
a) Predicting the exact temperature tomorrow  
b) Grouping emails as spam or not spam *  
c) Calculating the average of a list of numbers  
d) Compressing a file to save space

3. What does a feature mean in machine learning?  
a) A measurable input variable used by the model *  
b) The final answer the model predicts  
c) A mistake made during training  
d) A type of learning algorithm

4. Why do we usually split data into training and test sets?  
a) To make the dataset look larger  
b) To ensure the model memorizes the answers  
c) To delete duplicate records automatically  
d) To evaluate how well the model works on new data *

5. What is reinforcement learning mainly based on?

In [40]:
lines = [l.rstrip() for l in quiz.splitlines() if l.strip()]
questions = [l for l in lines if re.match(r"\d+\. ", l)]
option_lines = [l for l in lines if re.match(r"[a-d]\) ", l)]
stars = [l for l in option_lines if l.endswith("*")]
print("title ok:", lines[0] == "**Machine learning basics quiz**")
print("questions:", len(questions), "| options:", len(option_lines), "| starred answers:", len(stars))
print("correct letters:", [l[0] for l in stars])
display(Markdown(quiz))

title ok: True
questions: 5 | options: 20 | starred answers: 5
correct letters: ['c', 'b', 'a', 'd', 'a']


**Machine learning basics quiz**

1. What is a training dataset used for in machine learning?  
a) To test the model after it is built  
b) To store the final predictions  
c) To teach the model patterns from examples *  
d) To remove all errors from the data

2. Which task is an example of classification?  
a) Predicting the exact temperature tomorrow  
b) Grouping emails as spam or not spam *  
c) Calculating the average of a list of numbers  
d) Compressing a file to save space

3. What does a feature mean in machine learning?  
a) A measurable input variable used by the model *  
b) The final answer the model predicts  
c) A mistake made during training  
d) A type of learning algorithm

4. Why do we usually split data into training and test sets?  
a) To make the dataset look larger  
b) To ensure the model memorizes the answers  
c) To delete duplicate records automatically  
d) To evaluate how well the model works on new data *

5. What is reinforcement learning mainly based on?  
a) Learning from rewards and penalties *  
b) Learning only from labeled examples  
c) Learning without any data at all  
d) Learning by copying another model

**Note.** The model copies a format much better from a real example than from a description of it, so the prompt contains the full example quiz and the rules about the title, the numbering, the `a)` to `d)` markers and the `*` for the correct answer. The two trailing spaces on each line matter in Markdown (without them the options would run together in one paragraph), which is why they are spelled out. The check above counts the questions, the options and the starred answers to confirm the format came out identical.

#### 26. Create a comprehensive online course outline for beginners.

Write a prompt that generates entire course outline, following the example provided below. Ensure the prompt is detailed enough to cover all sections, with clear formatting and practical guidance. 

Sections to include in course outline:
- Course overview - description, objectives, target audience.
- Module breakdown - 5 modules with title, short description, three key learning outcomes and an interactive assignment in the end of each module.
- Assessment plan - a table outlining the assessment methods with description and weighting.
- Certification criteria - requirements to earn a certificate.
- Marketing strategy - step-by-step guide for promoting the course.

Include a detailed description of the type of information each section should contain. Maintain the **same formatting**, **section titles**, and **course name**. While the generated descriptions, modules and marketing strategy steps may differ and incorporate new ideas and concepts, it must remain aligned with the theme of the task.

Put the final output of your API call in a Markdown cell. 

#### Example:
---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---

In [41]:
COURSE_EXAMPLE = r'''---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---'''

PROMPT_26 = f"""You are an instructional designer. Write a complete beginner-level online course outline.

Keep the course name exactly: "Mastering Prompt Engineering for AI Applications".
Use exactly the same Markdown formatting and exactly the same section titles as the reference outline at the end of this message,
in the same order: Course Overview (with Objectives and Target Audience), Module Breakdown, Assessment Plan, Certification Criteria, Marketing Strategy.
Separate sections with horizontal rules (---) as in the reference.

What each section must contain:
- Course Overview: one paragraph describing the course, who it is for and what they gain; then Objectives (3 bullets) and Target Audience (3 bullets).
- Module Breakdown: exactly 5 modules, titled "Module N: <title>". Each module has a **Description:** (1-2 sentences), **Key Learning Outcomes:** (exactly 3, numbered) and one **Interactive Assignment:** (a concrete hands-on task). The modules must build from basics to a final project.
- Assessment Plan: a Markdown table with the columns Assessment Method, Description and Weightage (%). The weightages must add up to 100%.
- Certification Criteria: an introductory line, then 3 numbered requirements with clear numbers (a minimum score, required submissions, participation).
- Marketing Strategy: exactly 5 steps titled "Step N: <name>", each with 2-3 bullets of practical actions (for example social media, email, partnerships, content, reviews).

The wording of descriptions, modules and marketing steps may differ from the reference and can bring in new ideas, but it must stay on the theme of prompt engineering for beginners.
Reply with the outline only.

REFERENCE OUTLINE (copy its structure, not its sentences):
{COURSE_EXAMPLE}
"""

resp, secs = ask(PROMPT_26, reasoning_effort="none", temperature=0.7, max_completion_tokens=3500)
course_outline = texts(resp)[0]
print(f"finish_reason: {resp.choices[0].finish_reason} | output tokens: {resp.usage.completion_tokens} | {secs:.1f}s")

finish_reason: stop | output tokens: 1296 | 6.5s


In [42]:
# Quick format check against the requirements
required_titles = ["Course Outline: Mastering Prompt Engineering for AI Applications", "Course Overview", "Objectives", "Target Audience",
                   "Module Breakdown", "Assessment Plan", "Certification Criteria", "Marketing Strategy"]
print("missing section titles:", [t for t in required_titles if t not in course_outline] or "none")
print("modules:", len(re.findall(r"^#+\s*Module \d", course_outline, flags=re.M)),
      "| marketing steps:", len(re.findall(r"^#+\s*Step \d", course_outline, flags=re.M)),
      "| interactive assignments:", course_outline.count("Interactive Assignment"))
weights = [int(w) for w in re.findall(r"(\d+)\s*%", course_outline.split("Assessment Plan", 1)[-1].split("Certification Criteria", 1)[0])]
print("assessment weightages:", weights, "-> total", sum(weights))

with open("task26_course_outline.md", "w", encoding="utf-8") as f:
    f.write(course_outline)
print("\nSaved to task26_course_outline.md. Print it below and copy it into the Markdown cell underneath.\n")
print(course_outline)

missing section titles: none
modules: 5 | marketing steps: 5 | interactive assignments: 5
assessment weightages: [20, 35, 15, 25, 5] -> total 100

Saved to task26_course_outline.md. Print it below and copy it into the Markdown cell underneath.

---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-level course introduces learners to the fundamentals of prompt engineering and shows how to create clear, effective prompts for AI tools used in writing, research, analysis, and everyday productivity. It is designed for newcomers who want practical, hands-on skills they can apply immediately, whether they are students, professionals, creators, or curious beginners exploring AI. By the end of the course, learners will be able to design, test, and improve prompts with confidence and use them to get better results from AI applications.

#### Objectives
- Understand the core concepts and best practices of prompt engineering.
- Write an

_(Final output of the API call for task 26: run the two cells above, then replace this line with the printed outline. Double-click this cell to edit it, paste the text, and run the cell to render it.)_